# exp-back — ex1: exp_back — reuse cached out

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `exp-back`. Running the final beacon cell reports progress against the `Backprop: exp_back` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: exp_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`exp-back`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "exp-back"
DD_SUBTOPIC = "Backprop: exp_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `exp_back` — quick refresher

`exp(x)` is elementwise with local derivative `exp(x) = out`. Because `out` is already cached in the back-fn signature, we never need to recompute `exp`.

**Worked exemplar.**
```
out  = exp(x)              # forward
d/dx exp(x) = exp(x) = out # local derivative — use cached out
grad_in = grad_out * out   # chain rule
```

Same shape as `x`. Numerically cheaper than `grad_out * torch.exp(x)` because the exp was already paid for on the forward pass.

### Exercise 1 — exp_back — reuse cached out

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the elementwise chain rule to derive exp_back, reusing the cached `out` tensor instead of recomputing exp(x).
> Keywords: exp, elementwise, cached-out
> ```

**KCs targeted:** `chain-rule-elementwise`, `back-fn-uses-cached-out`

Implement `exp_back(grad_out, out, x)` for the forward op `out = exp(x)`.

Derivation:
- `d/dx exp(x) = exp(x) = out`.
- Chain rule: `dL/dx = grad_out * out`.

**Use the cached `out`** — do NOT call `torch.exp(x)` inside the backward fn. The whole reason the backward signature includes `out` is to skip recomputation.

Return a `torch.Tensor` with the same shape as `x`. No autograd.

In [ ]:
def exp_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d/dx exp(x) = exp(x) = out. Reuse the cached out — no recompute.
    return grad_out * out


<details><summary>Solution</summary>

```python
def exp_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d/dx exp(x) = exp(x) = out. Reuse the cached out — no recompute.
    return grad_out * out
```

**One multiplication, no exp call.** The forward already paid for the exponential. Reading it back from `out` saves one elementwise exp on every reverse pass.

**Why not `grad_out * t.exp(x)`.** Equivalent numerically (modulo float rounding) but costs one redundant exp per node. On long chains this is the difference between O(n) and O(2n) exp evaluations.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()